In [2]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


# Create wochenliste table

In [5]:
duck.sql(
    """
    create or replace table wochenliste as 
    select * replace("ID Nummer"::int64 as "ID Nummer") 
    from read_xlsx('/Users/adrienblanquer/Downloads/2026_KW06_Wochenliste.xlsm', sheet='Kunden', header=True, range='B5:D998')
    """)

# 1. Match easybill with medisoft

## 1.1 firt version of the match

In [6]:
duck.sql(
    """
    create or replace table easybill_x_medisoft as 
    with medisoft_data as (
        select 
            rec_id as medisoft_id,
            name,
            kuerzel,
            pfad,
            split(pfad, '/')[-1] as pfad_name,
            clean_account_name(split(pfad, '/')[-1]) as clean_pfad_name,
            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
    ), easybill_data as (
        select
            *,
            concat_ws(' ', "Kontakt: Straße/Hausnummer", "Kontakt: Postleitzahl", "Kontakt: Ort") as addresse,
            clean_account_name(coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name"))) as clean_firmenname
        from pg.easybill.contacts
    )
    select 
        eb."Kontakt: Kundennummer" as easybill_kundennummer,
        eb."Kontakt: Firma" as easybill_firma,
        eb."Kontakt: Name" as easybill_name,
        eb."Kontakt: Vorname" as easybill_vorname,
        eb.addresse as easybill_addresse,
        array_agg(medisoft_data.medisoft_id) as medisoft_ids,
        array_agg(medisoft_data.pfad_name) as medisoft_names,
        array_agg(
            jaro_winkler_similarity(medisoft_data.clean_pfad_name, eb.clean_firmenname)
        ) as sim
    from easybill_data as eb
    left join medisoft_data
        on jaro_winkler_similarity(medisoft_data.clean_pfad_name, eb.clean_firmenname) > 0.95
        --or jaro_winkler_similarity(medisoft_data.clean_kuerzel, eb.clean_firmenname) > 0.95
        --or jaro_winkler_similarity(medisoft_data.clean_name, eb.clean_firmenname) > 0.95
    group by eb."Kontakt: Kundennummer", eb."Kontakt: Firma", eb."Kontakt: Name", eb."Kontakt: Vorname", eb.addresse
    order by easybill_firma
    """
)

## 1.2 Second and last version

In [7]:
duck.sql(
    """
    create or replace table easybill_x_medisoft as 
    with medisoft_data as (
        select 
            rec_id as medisoft_id,
            name,
            kuerzel,
            pfad,
            split(pfad, '/')[-1] as pfad_name,
            clean_account_name(split(pfad, '/')[-1]) as clean_pfad_name,
            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
    ), 
    easybill_data as (
        select
            *,
            coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name")) as raw_firmenname,
            concat_ws(' ', "Kontakt: Straße/Hausnummer", "Kontakt: Postleitzahl", "Kontakt: Ort") as addresse,
            clean_account_name(coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name"))) as clean_firmenname
        from pg.easybill.contacts
    ),
    all_potential_matches as (
        -- Step 1: Find every possible match and calculate both sim scores
        select 
            eb."Kontakt: Kundennummer",
            eb."Kontakt: Firma",
            eb."Kontakt: Name",
            eb."Kontakt: Vorname",
            eb.addresse,
            ms.medisoft_id,
            ms.pfad,
            ms.pfad_name,
            jaro_winkler_similarity(ms.clean_pfad_name, eb.clean_firmenname) as sim_clean,
            jaro_winkler_similarity(ms.pfad_name, eb.raw_firmenname) as sim_raw
        from easybill_data as eb
        inner join medisoft_data as ms
            on jaro_winkler_similarity(ms.clean_pfad_name, eb.clean_firmenname) > 0.95
    ),
    best_unique_matches as (
        -- Step 2: Ensure each medisoft_id appears only ONCE globally
        -- We pick the best Easybill contact for each Medisoft record
        select * from all_potential_matches
        qualify row_number() over (
            partition by medisoft_id 
            order by sim_clean desc, sim_raw desc
        ) = 1
    )
    -- Step 3: Re-aggregate back to Easybill contacts
    select 
        eb."Kontakt: Kundennummer" as easybill_kundennummer,
        eb."Kontakt: Firma" as easybill_firma,
        eb."Kontakt: Name" as easybill_name,
        eb."Kontakt: Vorname" as easybill_vorname,
        eb.addresse as easybill_addresse,
        array_agg(m.medisoft_id) filter (where m.medisoft_id is not null) as medisoft_ids,
        array_agg(m.pfad) filter (where m.pfad is not null) as medisoft_names,
        array_agg(m.sim_clean) filter (where m.sim_clean is not null) as sim_scores
    from easybill_data eb
    left join best_unique_matches m 
        on eb."Kontakt: Kundennummer" = m."Kontakt: Kundennummer"
    group by 
        eb."Kontakt: Kundennummer", 
        eb."Kontakt: Firma", 
        eb."Kontakt: Name", 
        eb."Kontakt: Vorname", 
        eb.addresse
    order by easybill_firma
    """
)

## 1.3 attach wochenliste to the liste

In [8]:
duck.sql(
    """
    create or replace table easybill_x_medisoft_wochenliste as 
    select 
        easybill_x_medisoft.* replace(
            array_to_string(medisoft_ids, '\n') as medisoft_ids,
            array_to_string(medisoft_names, '\n') as medisoft_names,
            array_to_string(sim_scores, '\n') as sim_scores
        ),
        array_to_string(array_agg(wochenliste."ID Nummer"), '\n') as wochenliste_ids,
    from easybill_x_medisoft
    left join wochenliste
        on left(wochenliste."ID Nummer"::varchar, 9) = easybill_x_medisoft.easybill_kundennummer
    group by all
    order by easybill_firma
    """
)

## 1.4 attach document information to easybill data, also ZOHO

In [ ]:
duck.sql(
    """
    """
)

In [169]:
duck.sql(
    """
    with easybill_docs as (
        select * replace(
            replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer"
        )
        from read_csv('./data/*.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
    ), invoices_data as (
        select 
            replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer",
            max("Dokument: Datum") as last_invoice_date,
            sum(replace("Posten: Nettobetrag", ',', '.')::float) as total_net_billed,
            --sum(replace("Posten: Bruttobetrag", ',', '.')::float)
                    bool_or(
            (
                -- Condition A : Codes articles de type examens individuels
                ("Posten: Artikelnummer" LIKE 'AM-SSU%' OR "Posten: Artikelnummer" LIKE 'AM-GVS%')
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%Grundbetreuung%'
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%ASA-Sitzung%'
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%Begehung%'
            )
            OR 
            (
                -- Condition B : Mots clés médicaux techniques (même si le code est différent)
                -- On exclut les codes de gestion (AM-R, AS-R) et les frais de déplacement (AFPa)
                "Posten: Artikelbeschreibung" SIMILAR TO '.*(Untersuchung|Impfung|G2[4-6]|G4[1-2]|G37|FeV|P-Schein|Hörtest|Sehtest|Ergometrie|Labor).*'
                AND "Posten: Artikelnummer" NOT IN ('AFPa', 'AM-R', 'AS-R', 'Anlag', 'Sonst.', 'Sonstige')
            )
        ) as has_physical_medical_exams

        from easybill_docs
        group by "Kontakt: Kundennummer"
    )
    select 
        eb.easybill_kundennummer,
        inv.last_invoice_date,
        inv.total_net_billed::int as total_net_billed,
        inv.has_physical_medical_exams,
        eb.* exclude(easybill_kundennummer),
        zoho."Record ID"
    from easybill_x_medisoft_wochenliste as eb
    left join invoices_data as inv
        on replace(eb.easybill_kundennummer, ' ', '') = inv."Kontakt: Kundennummer"
    left join read_csv('/Users/adrienblanquer/Downloads/Accounts_2026_02_03.csv') as zoho
        on inv."Kontakt: Kundennummer" = zoho.Kundennummer
    """
).to_csv('output/easybill_firms_consolidated.csv')

## 1.5 easybill with zoho focus

In [62]:
duck.sql(
    """
    with easybill_docs as (
        select * replace(
            replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer"
        ), strip_accents(lower("Kontakt: Firma")) as clean_firmenname
        from read_csv('./data/*.csv', types={'Kontakt: Kundennummer': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR'})
        qualify row_number() over (partition by "Kontakt: Kontakt ID" order by length("Kontakt: Kundennummer") desc) = 1
    ),
    invoices_data as (
        select 
            replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer",
            max("Dokument: Datum") as last_invoice_date,
            sum(replace("Posten: Nettobetrag", ',', '.')::float) as total_net_billed,
            --sum(replace("Posten: Bruttobetrag", ',', '.')::float)
                    bool_or(
            (
                -- Condition A : Codes articles de type examens individuels
                ("Posten: Artikelnummer" LIKE 'AM-SSU%' OR "Posten: Artikelnummer" LIKE 'AM-GVS%')
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%Grundbetreuung%'
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%ASA-Sitzung%'
                AND "Posten: Artikelbeschreibung" NOT ILIKE '%Begehung%'
            )
            OR 
            (
                -- Condition B : Mots clés médicaux techniques (même si le code est différent)
                -- On exclut les codes de gestion (AM-R, AS-R) et les frais de déplacement (AFPa)
                "Posten: Artikelbeschreibung" SIMILAR TO '.*(Untersuchung|Impfung|G2[4-6]|G4[1-2]|G37|FeV|P-Schein|Hörtest|Sehtest|Ergometrie|Labor).*'
                AND "Posten: Artikelnummer" NOT IN ('AFPa', 'AM-R', 'AS-R', 'Anlag', 'Sonst.', 'Sonstige')
            )
        ) as has_physical_medical_exams
        from easybill_docs
        group by "Kontakt: Kundennummer"
    )
    select
        distinct on (d."Kontakt: Kundennummer")
        d."Kontakt: Kundennummer" as easybill_kundennummer,
        inv.last_invoice_date,
        replace(inv.total_net_billed::varchar, '.', ',') as total_net_billed,
        d."Kontakt: Firma" as easybill_firmenname,
        d."Kontakt: Name" as easybill_contact_name,
        d."Kontakt: Vorname" as easybill_contact_vorname,
        zoho."Accounts Name" as zoho_firmenname,
        case when d."Kontakt: Kundennummer" = zoho.Kundennummer then 1 else 0 end as id_match,
        'https://crm.zoho.eu/crm/org20078302933/tab/Accounts/' || right(zoho."Record ID", -5) as zoho_url
    from easybill_docs d
    join invoices_data as inv using("Kontakt: Kundennummer")
    left join read_csv('/Users/adrienblanquer/Downloads/Accounts_2026_02_03.csv') as zoho
        on d."Kontakt: Kundennummer" = zoho.Kundennummer
        or clean_firmenname = strip_accents(lower(zoho."Accounts Name"))
    order by d."Kontakt: Kundennummer"
    """
).to_csv('easybill_zoho_match.csv')

In [48]:
duck.sql("""    with easybill_docs as (
        select * replace(
            replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer"
        ), strip_accents(lower("Kontakt: Firma")) as clean_firmenname
        from read_csv('./data/*.csv', types={'Kontakt: Kundennummer': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR'})
    )
    select 
        distinct on ("Kontakt: Kundennummer")
        columns('^Kontakt*') 
    from read_csv('./data/*.csv', types={'Kontakt: Kundennummer': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR'})
    where "Kontakt: Firma" = 'FlexPhysio GbR' and length("Kontakt: Kundennummer") = 9
""")

┌─────────────────────┬───────────────────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────────────────────┬────────────────┬──────────────────┬───────────────┬───────────────────────┬────────────────┬────────────────────────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬────────────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────────────┬────────────────────┬────────────────────┬──────────────┬───────────────────────┬─────────────────┬──────────────────────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────────┬──────────────────────────────┬──────────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┬──────────────────────┬───────────────────────┬──────────────┬───────────────┬───────────────────┬───────────────────┬──────────────

In [12]:
duck.sql(
    """
    from read_csv('/Users/adrienblanquer/Downloads/Accounts_2026_02_03.csv') as zoho
    """
)

┌─────────────────────────┬─────────────────────────┬────────────────┬───────────────────────────────────────────────────────┬─────────────────┬────────────────────┬─────────────────┬───────────────────────┬──────────┬───────────┬─────────────────────────┬─────────────────┬─────────────────────────┬─────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────────┬─────────────────┬──────────────┬───────────────┬─────────────────────┬────────────────┬──────────────┬───────────────┬─────────────────┬──────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [9]:
duck.sql(
    "from easybill_x_medisoft"
)

┌───────────────────────┬──────────────────────────────────────────────┬───────────────┬──────────────────┬────────────────────────────────────────────┬────────────────────────────────┬──────────────────────────────────────────────────────────────┬────────────┐
│ easybill_kundennummer │                easybill_firma                │ easybill_name │ easybill_vorname │             easybill_addresse              │          medisoft_ids          │                        medisoft_names                        │ sim_scores │
│        varchar        │                   varchar                    │    varchar    │     varchar      │                  varchar                   │           varchar[]            │                          varchar[]                           │  double[]  │
├───────────────────────┼──────────────────────────────────────────────┼───────────────┼──────────────────┼────────────────────────────────────────────┼────────────────────────────────┼─────────────────────────────

In [163]:
duck.sql("""
select
"Posten: Artikelnummer",
"Posten: Artikelbeschreibung"
from read_csv('./data/*.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
group by 1, 2
order by 1
""").to_csv('output/easybill_list_of_articles.csv')

In [51]:
duck.sql(
    """
    with medisoft_ids as (select unnest(medisoft_ids) as id from easybill_x_medisoft )
    select distinct on (id) * from medisoft_ids where id is not null
    --select * from pg.medisoft.table_firmenstruktur where pfad ilike 'BSH Berlin%'
    """
)

┌───────────────┐
│      id       │
│    varchar    │
├───────────────┤
│ 00_9XN00KD869 │
│ 00_9XW00JGVKD │
│ 00_9OI00PSMNQ │
│ 00_8YI00J991V │
│ 00_9SC00HTPTH │
│ 00_9JI00G23QS │
│ 00_9ZK00K7UYV │
│ 00_93600OB52A │
│ 00_9OG00PJACM │
│ 00_99P00IQB16 │
│       ·       │
│       ·       │
│       ·       │
│ 00_94C00KPP42 │
│ 00_8R300KDX6Y │
│ 00_92D00W0YY3 │
│ 00_97C00RBUU0 │
│ 00_9EE00Z3Y38 │
│ 00_90E00I46T9 │
│ 00_9A900KB53A │
│ 00_9YM00KKQRZ │
│ 00_9DE00QL4TJ │
│ 00_A0S00RIHE7 │
├───────────────┤
│   1425 rows   │
│  (20 shown)   │
└───────────────┘

In [83]:
duck.sql(
    """
    --create or replace table easybill_x_medisoft as 
    with medisoft_data as (
        select 
            rec_id as medisoft_id,
            name,
            kuerzel,
            pfad,
            split(pfad, '/')[-1] as pfad_name,
            clean_account_name(split(pfad, '/')[-1]) as clean_pfad_name,
            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
    ), 
    easybill_data as (
        select
            *,
            coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name")) as raw_firmenname,
            concat_ws(' ', "Kontakt: Straße/Hausnummer", "Kontakt: Postleitzahl", "Kontakt: Ort") as addresse,
            clean_account_name(coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name"))) as clean_firmenname
        from pg.easybill.contacts
    ),
    all_potential_matches as (
        -- Step 1: Find every possible match and calculate both sim scores
        select 
            eb."Kontakt: Kundennummer",
            eb."Kontakt: Firma",
            eb."Kontakt: Name",
            eb."Kontakt: Vorname",
            eb.addresse,
            ms.medisoft_id,
            ms.pfad_name,
            jaro_winkler_similarity(ms.clean_pfad_name, eb.clean_firmenname) as sim_clean,
            jaro_winkler_similarity(ms.pfad_name, eb.raw_firmenname) as sim_raw
        from easybill_data as eb
        inner join medisoft_data as ms
            on jaro_winkler_similarity(ms.clean_pfad_name, eb.clean_firmenname) > 0.95
    ),
    best_unique_matches as (
        -- Step 2: Ensure each medisoft_id appears only ONCE globally
        -- We pick the best Easybill contact for each Medisoft record
        select * from all_potential_matches
        qualify row_number() over (
            partition by medisoft_id 
            order by sim_clean desc, sim_raw desc
        ) = 1
    )
    -- Step 3: Re-aggregate back to Easybill contacts
    select 
        eb."Kontakt: Kundennummer" as easybill_kundennummer,
        eb."Kontakt: Firma" as easybill_firma,
        eb."Kontakt: Name" as easybill_name,
        eb."Kontakt: Vorname" as easybill_vorname,
        eb.addresse as easybill_addresse,
        array_agg(m.medisoft_id) filter (where m.medisoft_id is not null) as medisoft_ids,
        array_agg(m.pfad_name) filter (where m.pfad_name is not null) as medisoft_names,
        array_agg(m.sim_clean) filter (where m.sim_clean is not null) as sim_scores
    from easybill_data eb
    left join best_unique_matches m 
        on eb."Kontakt: Kundennummer" = m."Kontakt: Kundennummer"
    group by 
        eb."Kontakt: Kundennummer", 
        eb."Kontakt: Firma", 
        eb."Kontakt: Name", 
        eb."Kontakt: Vorname", 
        eb.addresse
    order by easybill_firma
    """
).show(max_rows=1000)

┌───────────────────────┬───────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────┬───────────────────────────┬─────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────┐
│ easybill_kundennummer │                                  easybill_firma                                   │             easybill_name              │     easybill_vorname      │                        easybill_addresse                        │                                        medisoft_ids                                        │                                                                            medisoft_names               

# 2. Standort medisoft files

In [196]:
duck.sql(
    """
    select 
    distinct on(trim(split(pfad, '/')[1]))
    split(pfad, '/')[1], mandant,
    coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name
    from pg.medisoft.table_firmenstruktur
    order by 1
    """
).show(max_rows=1000)

┌─────────────────────────────────┬──────────────┬─────────────────┐
│       split(pfad, '/')[1]       │   mandant    │ clean_pfad_name │
│             varchar             │   varchar    │     varchar     │
├─────────────────────────────────┼──────────────┼─────────────────┤
│                                 │ Berlin       │ Berlin          │
│ BSH Berlin                      │ Berlin       │ Berlin          │
│ BSH Düsseldorf                  │ Düsseldorf   │ Düsseldorf      │
│ BSH Frankfurt                   │ Frankfurt    │ Frankfurt       │
│ BSH Hamburg                     │ Hamburg      │ Hamburg         │
│ BSH Kiel                        │ Kiel         │ Kiel            │
│ BSH Köln                        │ Köln         │ Köln            │
│ BSH München                     │ München      │ München         │
│ BSH Rostock                     │ Rostock      │ Rostock         │
│ BSH Stuttgart                   │ Stuttgart    │ Stuttgart       │
│ BSH Viersen                     

## 2.1 Berlin

In [241]:
duck.sql(
    """
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    berlin_firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%Berlin%'
    )
    select 
        berlin_firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', berlin_firms.strasse, berlin_firms.plz, berlin_firms.ort) as addresse,
    from berlin_firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = berlin_firms.rec_id or b.ebetrieb_id = berlin_firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = berlin_firms.rec_id or u.ebetrieb_id = berlin_firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 2
    """
).to_csv('output/medisoft_berlin_firms.csv')


## 2.2 Düsseldorf

In [239]:
city = 'Düsseldorf'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')


## 2.3 Frankfurt

In [238]:
city = 'Frankfurt'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')


## 2.4 Hamburg

In [235]:
city = 'Hamburg'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

## 2.5 Kiel

In [234]:
city = 'Kiel'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

## 2.6 Köln

In [242]:
city = 'Köln'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

## 2.7 München

In [10]:
city = 'München'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

## 2.8 Rostock

In [12]:
city = 'Rostock'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

# 2.9 Stuttgart

In [13]:
city = 'Stuttgart'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')

# 2.10 Viersen


In [14]:
city = 'Viersen'
duck.sql(
    f"""
    with parsed_pfad as (
        select
        coalesce(nullif(regexp_extract(split(pfad, '/')[1], 'BSH\\s(.*)', 1), ''),mandant) as clean_pfad_name,
        *
        from pg.medisoft.table_firmenstruktur
    ),
    firms as (
        select * from parsed_pfad
        where clean_pfad_name ilike '%{city}%'
    )
    select 
        firms.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum::date) as last_exam_date,
        concat_ws(' ', firms.strasse, firms.plz, firms.ort) as addresse,
    from firms
    left join pg.medisoft.table_beschaeftigte b
        on b.abetrieb_id = firms.rec_id or b.ebetrieb_id = firms.rec_id
    left join pg.medisoft.table_untersuchungen u
        on u.abetrieb_id = firms.rec_id or u.ebetrieb_id = firms.rec_id
    group by 1, 2, 3, 4, 7
    order by 1
    """
).to_csv(f'output/medisoft_{city}_firms.csv')